# Các ghi chú cho Team


## Sơ đồ xử lý pipeline về logic phân nhánh (Sprint 2)

```text
[ DỮ LIỆU CỦA TÙNG ] (File data.csv đã sạch cơ bản)
         │
         ▼
========================================================================
TRẠM 1: BIẾN ĐỔI DỮ LIỆU (File: processor.py)
⚙️ Công tắc config: 'target_transform'
  ├─ Nhánh A: RAW ──> (Giữ nguyên số tiền) ───────────┐
  └─ Nhánh B: LOG ──> (Dùng np.log1p ép về chuẩn) ────┤
                                                      │
                                             [ Dữ liệu chuẩn hóa ]
                                                      │
         ┌────────────────────────────────────────────┘
         ▼
========================================================================
TRẠM 2: LỰA CHỌN ĐẶC TRƯNG (File: feature_selector.py)  <-- (Task UT-71 của bạn)
⚙️ Công tắc config: 'feature_selection_method'
  ├─ Nhánh 1: Filter Method   (Lọc qua độ tương quan) ────────┐
  ├─ Nhánh 2: Wrapper Method  (Dùng thuật toán RFE) ──────────┼──> [ Dữ liệu ]
  └─ Nhánh 3: Embedded Method (Dùng Lasso/Tree Importance) ───┘    [ Tinh gọn]
                                                                        │
         ┌──────────────────────────────────────────────────────────────┘
         ▼
========================================================================
TRẠM 3: HUẤN LUYỆN (File: model.py)
⚙️ Công tắc config: 'model_params'
  └──> Model chỉ việc nhắm mắt học trên tệp dữ liệu đã được tinh gọn ở trên.
       (Output: Kết quả dự báo y_pred)
         │
         ▼
========================================================================
TRẠM 4: BÁO CÁO & ĐÁNH GIÁ (File: report.py)
⚙️ Hệ thống nhìn lại Công tắc Trạm 1
  ├─ Nếu Trạm 1 dùng LOG ──> Dùng expm1 để dịch ngược về số tiền thật.
  ├─ Nếu Trạm 1 dùng RAW ──> Giữ nguyên.
  └──> Xuất ra file baseline_report.json
```


## 2. Vai trò File trong hệ thống

Để dây chuyền trên hoạt động trơn tru, bạn cần phân rõ trách nhiệm cho từng file, không để file này làm hộ việc của file kia:

- **`config.yaml` (Bảng điều khiển trung tâm):** Nơi chứa tất cả các "Công tắc". Tùng hay Hoàng muốn thử nghiệm (đổi log/raw, đổi cách chọn feature, tinh chỉnh model), chỉ cần sửa thông số ở file này. Tuyệt đối hạn chế sửa code Python khi chỉ muốn test ý tưởng.
- **`main.py` (Nhạc trưởng):** File này KHÔNG chứa logic tính toán phức tạp. Nó chỉ làm nhiệm vụ điều phối và gọi các file khác theo đúng thứ tự:
- **`src/preprocess.py` (Trạm làm sạch - Tùng):** Nơi nhận file CSV thô gốc và thực hiện các bước dọn dẹp cơ bản (xử lý missing value, chuẩn hóa định dạng chữ, loại bỏ các cột rác tẻ nhạt). Đầu ra là dữ liệu sạch để đưa vào Pipeline động.
- **`src/processor.py` (Trạm biến đổi - Can):** Nơi chứa logic toán học để biến đổi dữ liệu một cách linh hoạt dựa theo config (ví dụ: tự động lấy Log cho features/target, hoặc Scaling dữ liệu).
- **`src/feature_selector.py` (Trạm gác cổng - Can):** Nơi chứa 3 thuật toán chọn feature (Filter, Wrapper, Embedded). Nó nhận vào hàng trăm cột, đọc config xem team muốn dùng cách nào, và tự động gọt dũa trả ra những cột tốt nhất.
- **`src/model.py` (Trạm huấn luyện - Hoàng):** Nơi định nghĩa các thuật toán Machine Learning. Trạm này chỉ việc "nhắm mắt" nhận tệp dữ liệu đã tinh gọn ở trên để huấn luyện và trả ra kết quả dự báo.
- **`src/report.py` (Trạm phiên dịch - Can):** Nơi chuyển đổi các con số dự báo khô khan (như dạng log) thành tiền thật, tính toán các chỉ số đánh giá (RMSE, R2_Score) và xuất ra file báo cáo `.json`.
